# Government Schemes — Feature Engineering & Dataset Integration

## Objective

This notebook integrates the Hugging Face and GitHub government schemes datasets into a unified AI-ready dataset for the Agentic AI government scheme discovery system.

The workflow includes:

- Standardizing the two datasets
- Identifying overlapping schemes
- Matching and merging scheme records
- Preserving complementary information from both sources
- Creating normalized government, geographic, category, beneficiary, and eligibility features
- Combining structured and unstructured eligibility information
- Creating a unified semantic representation for RAG
- Preserving source provenance
- Removing redundant and EDA-only features
- Validating the final dataset
- Saving the final AI-ready government schemes dataset

The final dataset will be designed to support both:
1. Structured filtering and agentic decision-making
2. Semantic retrieval and RAG-based question answering

In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)

In [2]:
# Load Hugging Face dataset
hf_df = pd.read_csv("/content/Schemes.csv")

# Load GitHub dataset
github_df = pd.read_csv("/content/gov_myscheme_data.csv")

print("Hugging Face dataset shape:", hf_df.shape)
print("GitHub dataset shape:", github_df.shape)

Hugging Face dataset shape: (4693, 24)
GitHub dataset shape: (2066, 16)


In [3]:
print("HUGGING FACE DATASET")
print("=" * 50)
print(hf_df.columns.tolist())

print("\nGITHUB DATASET")
print("=" * 50)
print(github_df.columns.tolist())

HUGGING FACE DATASET
['slug', 'name', 'description', 'ministry', 'department', 'state', 'category', 'beneficiary_type', 'benefits', 'eligibility_text', 'application_process', 'documents_required', 'apply_url', 'official_url', 'eligibility_age_min', 'eligibility_age_max', 'eligibility_gender', 'eligibility_caste', 'eligibility_income_max', 'eligibility_residence', 'eligibility_state', 'eligibility_disability', 'eligibility_bpl', 'scraped_at']

GITHUB DATASET
['Scheme Name', 'Scheme Slug', 'Level', 'State / UT / Ministry', 'Application Mode', 'Tags / Categories', 'Description', 'Eligibility Criteria', 'Eligibility (General)', 'Exclusions / Ineligibility', 'Benefits', 'Application Process', 'Documents Required', 'Frequently Asked Questions (FAQs)', 'Official Link', 'MyScheme URL']


In [5]:
hf_slugs = set(hf_df["slug"].dropna().astype(str).str.strip().str.lower())
github_slugs = set(github_df["Scheme Slug"].dropna().astype(str).str.strip().str.lower())

common_slugs = hf_slugs.intersection(github_slugs)

print("Hugging Face schemes:", len(hf_slugs))
print("GitHub schemes:", len(github_slugs))
print("Exact slug matches:", len(common_slugs))
print("HF-only schemes:", len(hf_slugs - github_slugs))
print("GitHub-only schemes:", len(github_slugs - hf_slugs))

Hugging Face schemes: 4693
GitHub schemes: 2066
Exact slug matches: 1895
HF-only schemes: 2798
GitHub-only schemes: 171


In [6]:
matched_slugs = list(common_slugs)

slug_match_df = pd.merge(
    hf_df[["slug", "name"]],
    github_df[["Scheme Slug", "Scheme Name"]],
    left_on=hf_df["slug"].astype(str).str.strip().str.lower(),
    right_on=github_df["Scheme Slug"].astype(str).str.strip().str.lower(),
    how="inner"
)

slug_match_df = slug_match_df.rename(columns={
    "name": "HF Name",
    "Scheme Name": "GitHub Name"
})

slug_match_df[["HF Name", "GitHub Name", "slug", "Scheme Slug"]].head(20)

,HF Name,GitHub Name,slug,Scheme Slug
0,"""Immediate Relief Assistance"" under ""Welfare and Relief for Fishermen During Lean Seasons and N...","""Immediate Relief Assistance"" under ""Welfare and Relief for Fishermen During Lean Seasons and Na...",ira-wrflsncs,ira-wrflsncs
1,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,AICTE SHORT TERM TRAINING PROGRAMME-SFURTI SCHEME,astpss,astpss
2,Consortia & Tender Marketing Scheme,Consortia & Tender Marketing Scheme,ctms,ctms
3,Indira Mahila Shakti Udyam Protsahan Yojana,Indira Mahila Shakti Udyam Protsahan Yojana,imspesy,imspesy
4,Nirman Shramik Jeevan va Bhavishya Suraksha Yojana,Nirman Shramik Jeevan Bhavishya Suraksha Yojana,nsjbsy,nsjbsy
5,Safai Karmakaar Ke Putr/Putri Hetu Vishesh Coaching Yojana,Safai Karmakaar Ke Putr/Putri Hetu Vishesh Coaching Yojana,skhvc,skhvc
6,"'Financial Assistance for Setting Up of Sales Emporia by Industrial Associations, Handicrafts So...","'Financial Assistance for Setting Up of Sales Emporia by Industrial Associations, Handicrafts So...",fasuseiahsri-mp,fasuseiahsri-mp
7,"'Grant-in-Aid to Industrial Association, CII, Industrial NGOs for Promotion of SME Clusters' und...","'Grant-in-Aid to Industrial Association, CII, Industrial NGOs for Promotion of SME Clusters' und...",giaiacinpsc,giaiacinpsc
8,"'Organizing/ Participation in Exhibitions /Trade Fairs /Expo /Seminars /Investors Meets, etc' un...","'Organizing/ Participation in Exhibitions /Trade Fairs /Expo /Seminars /Investors Meets, etc' un...",opetfesime-mp,opetfesime-mp
9,'Rent Subsidy for Sales Emporia Established by Handicrafts / Micro & Small Industries Emporium' ...,'Rent Subsidy for Sales Emporia Established by Handicrafts / Micro & Small Industries Emporium' ...,rsseehmsie-mp,rsseehmsie-mp


In [7]:
import re

def normalize_scheme_name(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    # Normalize common punctuation
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Normalize whitespace
    text = re.sub(r"\s+", " ", text).strip()

    return text


hf_df["name_normalized"] = hf_df["name"].apply(normalize_scheme_name)
github_df["name_normalized"] = github_df["Scheme Name"].apply(normalize_scheme_name)

print("HF normalized names created:", hf_df["name_normalized"].notna().sum())
print("GitHub normalized names created:", github_df["name_normalized"].notna().sum())

HF normalized names created: 4693
GitHub normalized names created: 2066


In [8]:
# Identify schemes that did not match by slug
hf_unmatched = hf_df[
    ~hf_df["slug"].astype(str).str.strip().str.lower().isin(common_slugs)
].copy()

github_unmatched = github_df[
    ~github_df["Scheme Slug"].astype(str).str.strip().str.lower().isin(common_slugs)
].copy()

# Find exact matches using normalized scheme names
name_matches = pd.merge(
    hf_unmatched[["slug", "name", "name_normalized"]],
    github_unmatched[["Scheme Slug", "Scheme Name", "name_normalized"]],
    on="name_normalized",
    how="inner"
)

print("HF-only records:", len(hf_unmatched))
print("GitHub-only records:", len(github_unmatched))
print("Additional exact normalized-name matches:", len(name_matches))

name_matches[[
    "name",
    "Scheme Name",
    "slug",
    "Scheme Slug"
]].head(20)

HF-only records: 2798
GitHub-only records: 171
Additional exact normalized-name matches: 2


,name,Scheme Name,slug,Scheme Slug
0,Devnarayan Gurukul Yojana,Devnarayan Gurukul Yojana,dgyr,dgy
1,National Agriculture Insurance Scheme,National Agriculture Insurance Scheme,nai,nais


In [9]:
# Create confirmed scheme matches

slug_matches = pd.DataFrame({
    "hf_slug": list(common_slugs),
    "github_slug": list(common_slugs),
    "match_type": "exact_slug"
})

name_matches_confirmed = name_matches.rename(columns={
    "slug": "hf_slug",
    "Scheme Slug": "github_slug"
})[[
    "hf_slug",
    "github_slug"
]].copy()

name_matches_confirmed["match_type"] = "exact_normalized_name"

confirmed_matches = pd.concat(
    [slug_matches, name_matches_confirmed],
    ignore_index=True
).drop_duplicates(subset=["hf_slug", "github_slug"])

print("Confirmed matched scheme pairs:", len(confirmed_matches))
print("\nMatch types:")
print(confirmed_matches["match_type"].value_counts())

Confirmed matched scheme pairs: 1897

Match types:
match_type
exact_slug               1895
exact_normalized_name       2
Name: count, dtype: int64


In [10]:
# Inspect information availability for the confirmed overlapping schemes

hf_matched = hf_df[
    hf_df["slug"].astype(str).str.strip().str.lower().isin(
        confirmed_matches["hf_slug"]
    )
].copy()

github_matched = github_df[
    github_df["Scheme Slug"].astype(str).str.strip().str.lower().isin(
        confirmed_matches["github_slug"]
    )
].copy()

print("Matched HF records:", len(hf_matched))
print("Matched GitHub records:", len(github_matched))

print("\nHF columns with missing values:")
print(hf_matched.isna().sum().sort_values(ascending=False))

print("\nGitHub columns with missing values:")
print(github_matched.isna().sum().sort_values(ascending=False))

Matched HF records: 1897
Matched GitHub records: 1897

HF columns with missing values:
eligibility_age_max       1553
eligibility_income_max    1549
ministry                  1451
eligibility_age_min       1431
department                 167
beneficiary_type            43
apply_url                    5
documents_required           4
benefits                     1
description                  1
eligibility_text             1
application_process          1
category                     1
state                        0
name                         0
slug                         0
official_url                 0
eligibility_gender           0
eligibility_caste            0
eligibility_residence        0
eligibility_state            0
eligibility_disability       0
eligibility_bpl              0
scraped_at                   0
name_normalized              0
dtype: int64

GitHub columns with missing values:
Exclusions / Ineligibility           1564
Tags / Categories                       5
Docu

In [11]:
github_clean = github_df.rename(columns={
    "Scheme Name": "scheme_name",
    "Scheme Slug": "scheme_slug",
    "Level": "government_level",
    "State / UT / Ministry": "state_ut_ministry",
    "Application Mode": "application_mode",
    "Tags / Categories": "raw_tags",
    "Description": "description",
    "Eligibility Criteria": "eligibility_criteria",
    "Eligibility (General)": "eligibility_general",
    "Exclusions / Ineligibility": "exclusion_text",
    "Benefits": "benefits",
    "Application Process": "application_process",
    "Documents Required": "documents_required",
    "Frequently Asked Questions (FAQs)": "faqs",
    "Official Link": "official_url_github",
    "MyScheme URL": "myscheme_url"
}).copy()

print("GitHub columns after standardization:")
print(github_clean.columns.tolist())

GitHub columns after standardization:
['scheme_name', 'scheme_slug', 'government_level', 'state_ut_ministry', 'application_mode', 'raw_tags', 'description', 'eligibility_criteria', 'eligibility_general', 'exclusion_text', 'benefits', 'application_process', 'documents_required', 'faqs', 'official_url_github', 'myscheme_url', 'name_normalized']


In [12]:
hf_clean = hf_df.rename(columns={
    "slug": "scheme_slug",
    "name": "scheme_name",
    "description": "description",
    "ministry": "ministry_hf",
    "department": "department",
    "state": "state_hf",
    "category": "category_hf",
    "beneficiary_type": "beneficiary_type_hf",
    "benefits": "benefits_hf",
    "eligibility_text": "eligibility_text_hf",
    "application_process": "application_process_hf",
    "documents_required": "documents_required_hf",
    "apply_url": "apply_url",
    "official_url": "official_url_hf",
    "eligibility_age_min": "age_min",
    "eligibility_age_max": "age_max",
    "eligibility_gender": "gender_eligibility",
    "eligibility_caste": "caste_category",
    "eligibility_income_max": "income_max",
    "eligibility_residence": "residence_requirement",
    "eligibility_state": "eligibility_state",
    "eligibility_disability": "disability_eligibility",
    "eligibility_bpl": "bpl_required",
    "scraped_at": "scraped_at"
}).copy()

print("Hugging Face columns after standardization:")
print(hf_clean.columns.tolist())

Hugging Face columns after standardization:
['scheme_slug', 'scheme_name', 'description', 'ministry_hf', 'department', 'state_hf', 'category_hf', 'beneficiary_type_hf', 'benefits_hf', 'eligibility_text_hf', 'application_process_hf', 'documents_required_hf', 'apply_url', 'official_url_hf', 'age_min', 'age_max', 'gender_eligibility', 'caste_category', 'income_max', 'residence_requirement', 'eligibility_state', 'disability_eligibility', 'bpl_required', 'scraped_at', 'name_normalized']


In [13]:
# Create clean scheme IDs for merging

hf_clean["scheme_id"] = (
    hf_clean["scheme_slug"]
    .astype(str)
    .str.strip()
    .str.lower()
)

github_clean["scheme_id"] = (
    github_clean["scheme_slug"]
    .astype(str)
    .str.strip()
    .str.lower()
)

# Check uniqueness
print("HF unique scheme IDs:", hf_clean["scheme_id"].nunique())
print("HF total records:", len(hf_clean))

print("\nGitHub unique scheme IDs:", github_clean["scheme_id"].nunique())
print("GitHub total records:", len(github_clean))

HF unique scheme IDs: 4693
HF total records: 4693

GitHub unique scheme IDs: 2066
GitHub total records: 2066


In [14]:
merged_df = hf_clean.merge(
    github_clean,
    on="scheme_id",
    how="outer",
    suffixes=("_hf", "_github"),
    indicator=True
)

print("Merged dataset shape:", merged_df.shape)

print("\nMerge status:")
print(merged_df["_merge"].value_counts())

Merged dataset shape: (4864, 44)

Merge status:
_merge
left_only     2798
both          1895
right_only     171
Name: count, dtype: int64


In [15]:
matched_preview = merged_df[
    merged_df["_merge"] == "both"
][[
    "scheme_id",
    "scheme_name_hf",
    "scheme_name_github",
    "scheme_slug_hf",
    "scheme_slug_github"
]].head(20)

matched_preview

,scheme_id,scheme_name_hf,scheme_name_github,scheme_slug_hf,scheme_slug_github
3,25-ciss,25% Capital Investment Subsidy Scheme,25% Capital Investment Subsidy Scheme,25-ciss,25-ciss
4,40shydcs,"40% Subsidy On Hank Yarn, Dyes & Chemicals Scheme","40% Subsidy On Hank Yarn, Dyes & Chemicals Scheme",40shydcs,40shydcs
8,a-gainer,AICTE - Grant For Augmenting Infrastructure In North Eastern Region,AICTE - Grant For Augmenting Infrastructure In North Eastern Region,a-gainer,a-gainer
9,a-pudu,ARAVANAIPPU,ARAVANAIPPU,a-pudu,a-pudu
10,a2ks,Access to Knowledge for Technology Development and Dissemination (A2K+),Access To Knowledge For Technology Development And Dissemination (A2K+) - Studies,a2ks,a2ks
13,aabcs,Annal Ambedkar Business Champions Scheme,Annal Ambedkar Business Champions Scheme,aabcs,aabcs
15,aaby,Aam Aadmi Bima Yojana,Aam Aadmi Bima Yojana,aaby,aaby
18,aaelss,Assam Abhinandan Education Loan Subsidy Scheme,Assam Abhinandan Education Loan Subsidy Scheme,aaelss,aaelss
20,aag,Aponar Apon Ghar,Aponar Apon Ghar,aag,aag
22,aamgsiscs,Award of Adhoc Merit Grant (Special Incentive) to Scheduled Caste Students,Award of Adhoc Merit Grant (Special Incentive) to Scheduled Caste Students,aamgsiscs,aamgsiscs


In [16]:
name_match_ids = ["dgyr", "nai"]

name_match_review = hf_clean[
    hf_clean["scheme_id"].isin(name_match_ids)
][[
    "scheme_id",
    "scheme_name",
    "scheme_slug",
    "name_normalized"
]].merge(
    github_clean[[
        "scheme_id",
        "scheme_name",
        "scheme_slug",
        "name_normalized"
    ]],
    on="name_normalized",
    how="inner",
    suffixes=("_hf", "_github")
)

name_match_review

,scheme_id_hf,scheme_name_hf,scheme_slug_hf,name_normalized,scheme_id_github,scheme_name_github,scheme_slug_github
0,dgyr,Devnarayan Gurukul Yojana,dgyr,devnarayan gurukul yojana,dgy,Devnarayan Gurukul Yojana,dgy
1,nai,National Agriculture Insurance Scheme,nai,national agriculture insurance scheme,nais,National Agriculture Insurance Scheme,nais


In [17]:
name_match_mapping = {
    "dgy": "dgyr",
    "nais": "nai"
}

print("Confirmed name-based matches:", len(name_match_mapping))
print(name_match_mapping)

Confirmed name-based matches: 2
{'dgy': 'dgyr', 'nais': 'nai'}


In [18]:
# Create unified IDs for the two confirmed name-based matches

github_clean["unified_scheme_id"] = github_clean["scheme_id"].replace(
    name_match_mapping
)

hf_clean["unified_scheme_id"] = hf_clean["scheme_id"]

print("HF unique unified IDs:", hf_clean["unified_scheme_id"].nunique())
print("GitHub unique unified IDs:", github_clean["unified_scheme_id"].nunique())

HF unique unified IDs: 4693
GitHub unique unified IDs: 2066


In [19]:
merged_df = hf_clean.merge(
    github_clean,
    on="unified_scheme_id",
    how="outer",
    suffixes=("_hf", "_github"),
    indicator=True
)

print("Unified merged dataset shape:", merged_df.shape)

print("\nMerge status:")
print(merged_df["_merge"].value_counts())

Unified merged dataset shape: (4862, 46)

Merge status:
_merge
left_only     2796
both          1897
right_only     169
Name: count, dtype: int64


In [20]:
print("Number of columns:", len(merged_df.columns))

for i, col in enumerate(merged_df.columns, start=1):
    print(f"{i:02d}. {col}")

Number of columns: 46
01. scheme_slug_hf
02. scheme_name_hf
03. description_hf
04. ministry_hf
05. department
06. state_hf
07. category_hf
08. beneficiary_type_hf
09. benefits_hf
10. eligibility_text_hf
11. application_process_hf
12. documents_required_hf
13. apply_url
14. official_url_hf
15. age_min
16. age_max
17. gender_eligibility
18. caste_category
19. income_max
20. residence_requirement
21. eligibility_state
22. disability_eligibility
23. bpl_required
24. scraped_at
25. name_normalized_hf
26. scheme_id_hf
27. unified_scheme_id
28. scheme_name_github
29. scheme_slug_github
30. government_level
31. state_ut_ministry
32. application_mode
33. raw_tags
34. description_github
35. eligibility_criteria
36. eligibility_general
37. exclusion_text
38. benefits
39. application_process
40. documents_required
41. faqs
42. official_url_github
43. myscheme_url
44. name_normalized_github
45. scheme_id_github
46. _merge


In [21]:
# Canonical scheme name
merged_df["scheme_name"] = (
    merged_df["scheme_name_hf"]
    .combine_first(merged_df["scheme_name_github"])
)

# Canonical scheme slug
merged_df["scheme_slug"] = (
    merged_df["scheme_slug_hf"]
    .combine_first(merged_df["scheme_slug_github"])
)

# Unified scheme ID is already our canonical merge identifier
merged_df["scheme_id"] = merged_df["unified_scheme_id"]

print("Missing scheme names:", merged_df["scheme_name"].isna().sum())
print("Missing scheme slugs:", merged_df["scheme_slug"].isna().sum())
print("Missing scheme IDs:", merged_df["scheme_id"].isna().sum())

print("\nUnique scheme IDs:", merged_df["scheme_id"].nunique())
print("Total records:", len(merged_df))

Missing scheme names: 0
Missing scheme slugs: 0
Missing scheme IDs: 0

Unique scheme IDs: 4862
Total records: 4862


In [23]:
# Normalize GitHub government level
government_level_github = (
    merged_df["government_level"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Start with the explicit GitHub value
merged_df["government_level_unified"] = government_level_github.replace({
    "state/ut": "State/UT",
    "central": "Central"
})

# Normalize HF state field
hf_state = (
    merged_df["state_hf"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Derive government level only for records where GitHub level is missing
hf_derived_level = pd.Series(pd.NA, index=merged_df.index, dtype="string")

hf_derived_level.loc[hf_state == "central"] = "Central"
hf_derived_level.loc[
    hf_state.notna() & (hf_state != "central")
] = "State/UT"

# Fill only missing GitHub-derived values
merged_df["government_level_unified"] = (
    merged_df["government_level_unified"]
    .fillna(hf_derived_level)
)

print(merged_df["government_level_unified"].value_counts(dropna=False))

government_level_unified
State/UT                 2559
state / ut government    1539
central government        527
Central                   237
Name: count, dtype: Int64


In [24]:
# Normalize government level values from GitHub
government_level_github = (
    merged_df["government_level"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Map all observed variants to a single standard representation
merged_df["government_level_unified"] = government_level_github.map({
    "state/ut": "State/UT",
    "state / ut government": "State/UT",
    "central": "Central",
    "central government": "Central"
})

# For HF-only records, derive the level from state_hf
hf_state = (
    merged_df["state_hf"]
    .astype("string")
    .str.strip()
    .str.lower()
)

hf_derived_level = pd.Series(
    pd.NA,
    index=merged_df.index,
    dtype="string"
)

hf_derived_level.loc[hf_state == "central"] = "Central"
hf_derived_level.loc[
    hf_state.notna() & (hf_state != "central")
] = "State/UT"

# Fill missing GitHub values using HF-derived values
merged_df["government_level_unified"] = (
    merged_df["government_level_unified"]
    .fillna(hf_derived_level)
)

print(merged_df["government_level_unified"].value_counts(dropna=False))

government_level_unified
State/UT    4098
Central      764
Name: count, dtype: int64


In [25]:
# Create unified State/UT field
github_state_value = (
    merged_df["state_ut_ministry"]
    .astype("string")
    .str.strip()
)

hf_state_value = (
    merged_df["state_hf"]
    .astype("string")
    .str.strip()
)

# Use GitHub state value for State/UT schemes
merged_df["state_ut"] = pd.NA

state_mask = merged_df["government_level_unified"] == "State/UT"

merged_df.loc[state_mask, "state_ut"] = (
    github_state_value[state_mask]
    .fillna(hf_state_value[state_mask])
)

# For Central schemes, state_ut should remain missing
merged_df["state_ut"] = merged_df["state_ut"].astype("string")

print("Missing state_ut:", merged_df["state_ut"].isna().sum())

print("\nState/UT values:")
print(merged_df.loc[state_mask, "state_ut"].value_counts().head(40))

Missing state_ut: 764

State/UT values:
state_ut
Gujarat                                     639
Uttarakhand                                 448
Madhya Pradesh                              286
Goa                                         272
Puducherry                                  270
Haryana                                     246
Tamil Nadu                                  232
Rajasthan                                   158
Bihar                                       113
West Bengal                                 112
Chhattisgarh                                107
Jharkhand                                    95
Odisha                                       89
Maharashtra                                  85
Meghalaya                                    84
Assam                                        82
Kerala                                       81
Himachal Pradesh                             71
Andhra Pradesh                               65
Karnataka                              

In [26]:
# Create unified ministry field
hf_ministry = (
    merged_df["ministry_hf"]
    .astype("string")
    .str.strip()
)

github_ministry = (
    merged_df["state_ut_ministry"]
    .astype("string")
    .str.strip()
)

merged_df["ministry"] = hf_ministry

# For missing ministry values, use the GitHub value only for Central schemes
central_mask = merged_df["government_level_unified"] == "Central"

merged_df.loc[central_mask, "ministry"] = (
    merged_df.loc[central_mask, "ministry"]
    .fillna(github_ministry[central_mask])
)

merged_df["ministry"] = merged_df["ministry"].astype("string")

print("Missing ministry:", merged_df["ministry"].isna().sum())

print("\nTop ministry values:")
print(merged_df["ministry"].value_counts().head(20))

Missing ministry: 4098

Top ministry values:
ministry
Ministry of Education                                  90
Ministry Of Social Justice and Empowerment             84
Ministry Of Science And Technology                     64
Ministry Of Commerce And Industry                      47
Ministry Of Agriculture and Farmers Welfare            41
Ministry Of Micro, Small and Medium Enterprises        30
Ministry of Social Justice and Empowerment             27
Ministry Of Textiles                                   24
Ministry of Electronics and Information Technology     24
Ministry Of Home Affairs                               18
Ministry Of Finance                                    18
Ministry Of Culture                                    15
Ministry Of Health & Family Welfare                    15
Ministry Of Defence                                    15
Ministry of Women and Child Development                13
Ministry Of Labour and Employment                      13
Ministry of Fisher

In [27]:
hf_ministry = merged_df["ministry_hf"].astype("string").str.strip()
github_ministry = merged_df["state_ut_ministry"].astype("string").str.strip()
central_mask = merged_df["government_level_unified"] == "Central"
merged_df["ministry"] = hf_ministry
merged_df.loc[central_mask, "ministry"] = merged_df.loc[central_mask, "ministry"].fillna(github_ministry[central_mask])
merged_df["ministry"] = merged_df["ministry"].astype("string")

print("Missing ministry:", merged_df["ministry"].isna().sum())
print("\nTop ministry values:")
print(merged_df["ministry"].value_counts().head(20))

Missing ministry: 4098

Top ministry values:
ministry
Ministry of Education                                  90
Ministry Of Social Justice and Empowerment             84
Ministry Of Science And Technology                     64
Ministry Of Commerce And Industry                      47
Ministry Of Agriculture and Farmers Welfare            41
Ministry Of Micro, Small and Medium Enterprises        30
Ministry of Social Justice and Empowerment             27
Ministry Of Textiles                                   24
Ministry of Electronics and Information Technology     24
Ministry Of Home Affairs                               18
Ministry Of Finance                                    18
Ministry Of Culture                                    15
Ministry Of Health & Family Welfare                    15
Ministry Of Defence                                    15
Ministry of Women and Child Development                13
Ministry Of Labour and Employment                      13
Ministry of Fisher

In [28]:
merged_df["ministry"] = merged_df["ministry"].str.replace(r"\s+", " ", regex=True).str.strip()
print("Central schemes with ministry:", merged_df.loc[central_mask, "ministry"].notna().sum())
print("Central schemes without ministry:", merged_df.loc[central_mask, "ministry"].isna().sum())

Central schemes with ministry: 762
Central schemes without ministry: 2


In [29]:
print(merged_df.loc[central_mask & merged_df["ministry"].isna(), ["scheme_name", "state_ut_ministry", "scheme_slug_github"]])

                                                                            scheme_name  \
4708  Vanchit Ikai Samooh aur Vargon ki Aarthik Sahayata Yojana (VISVAS) for Individual   
4718                                                      Varishtha Pension Bima Yojana   

     state_ut_ministry scheme_slug_github  
4708               NaN                NaN  
4718               NaN                NaN  


In [30]:
print(merged_df.loc[central_mask & merged_df["ministry"].isna(), ["scheme_name", "scheme_slug_hf", "description_hf", "ministry_hf"]].to_string(index=False))

                                                                      scheme_name scheme_slug_hf description_hf ministry_hf
Vanchit Ikai Samooh aur Vargon ki Aarthik Sahayata Yojana (VISVAS) for Individual        visvasi            NaN         NaN
                                                    Varishtha Pension Bima Yojana           vpby            NaN         NaN


In [31]:
merged_df["department_unified"] = merged_df["department"].astype("string").str.strip()
print("Missing department:", merged_df["department_unified"].isna().sum())
print("Unique departments:", merged_df["department_unified"].nunique())

Missing department: 462
Unique departments: 376


In [32]:
print(merged_df["department_unified"].value_counts().head(20))

department_unified
Labour Department                                           266
Agriculture, Farmers Welfare and Cooperation Department     222
Social Justice and Empowerment Department                   210
Agriculture Department                                      173
Social Welfare Department                                   130
Industries and Mines Department                             116
Labour and Employment Department                             84
Art, Culture and Museums Department                          80
Industries and Commerce Department, Puducherry               74
Department Of Higher Education                               71
Women and Child Development Department                       67
Fisheries Department                                         62
Tourism Department                                           61
Education Department                                         59
Higher Education Department                                  57
Social Justice and Sp

In [33]:
print(merged_df["department_unified"].nunique())
print(merged_df["department_unified"].isna().sum())

376
462


In [34]:
print(merged_df["department_unified"].value_counts().head(10))

department_unified
Labour Department                                          266
Agriculture, Farmers Welfare and Cooperation Department    222
Social Justice and Empowerment Department                  210
Agriculture Department                                     173
Social Welfare Department                                  130
Industries and Mines Department                            116
Labour and Employment Department                            84
Art, Culture and Museums Department                         80
Industries and Commerce Department, Puducherry              74
Department Of Higher Education                              71
Name: count, dtype: Int64


In [35]:
merged_df["department_unified"] = merged_df["department_unified"].replace(r"\s+", " ", regex=True).str.strip()
print(merged_df["department_unified"].nunique(), merged_df["department_unified"].isna().sum())

376 462


In [36]:
merged_df["category_unified"] = merged_df["category_hf"].astype("string").str.strip()
print("Missing category:", merged_df["category_unified"].isna().sum())
print("Unique categories:", merged_df["category_unified"].nunique())

Missing category: 173
Unique categories: 279


In [37]:
print(merged_df["category_unified"].value_counts().head(20))

category_unified
Education & Learning                                                     727
Social welfare & Empowerment                                             686
Agriculture,Rural & Environment                                          667
Business & Entrepreneurship                                              417
Social welfare & Empowerment, Women and Child                            155
Sports & Culture                                                         154
Skills & Employment                                                      127
Health & Wellness                                                        124
Education & Learning, Social welfare & Empowerment                        87
Banking,Financial Services and Insurance                                  72
Housing & Shelter                                                         65
Banking,Financial Services and Insurance, Business & Entrepreneurship     56
Education & Learning, Women and Child                      

In [38]:
category_map = {"Agriculture,Rural & Environment": "Agriculture & Rural Development", "Social welfare & Empowerment": "Social Welfare & Empowerment", "Banking,Financial Services and Insurance": "Banking, Financial Services & Insurance"}
merged_df["category_unified"] = merged_df["category_unified"].replace(category_map)
print(merged_df["category_unified"].value_counts().head(20))

category_unified
Education & Learning                                                     727
Social Welfare & Empowerment                                             686
Agriculture & Rural Development                                          667
Business & Entrepreneurship                                              417
Social welfare & Empowerment, Women and Child                            155
Sports & Culture                                                         154
Skills & Employment                                                      127
Health & Wellness                                                        124
Education & Learning, Social welfare & Empowerment                        87
Banking, Financial Services & Insurance                                   72
Housing & Shelter                                                         65
Banking,Financial Services and Insurance, Business & Entrepreneurship     56
Education & Learning, Women and Child                      

In [39]:
merged_df["category_unified"] = merged_df["category_unified"].str.replace("Social welfare & Empowerment", "Social Welfare & Empowerment", regex=False).str.replace("Agriculture,Rural & Environment", "Agriculture & Rural Development", regex=False).str.replace("Banking,Financial Services and Insurance", "Banking, Financial Services & Insurance", regex=False).str.replace(", ", ", ", regex=False)
print(merged_df["category_unified"].value_counts().head(20))

category_unified
Education & Learning                                                    727
Social Welfare & Empowerment                                            686
Agriculture & Rural Development                                         667
Business & Entrepreneurship                                             417
Social Welfare & Empowerment, Women and Child                           155
Sports & Culture                                                        154
Skills & Employment                                                     127
Health & Wellness                                                       124
Education & Learning, Social Welfare & Empowerment                       87
Banking, Financial Services & Insurance                                  72
Housing & Shelter                                                        65
Banking, Financial Services & Insurance, Business & Entrepreneurship     56
Education & Learning, Women and Child                                  

In [40]:
merged_df["beneficiary_groups"] = merged_df["beneficiary_type_hf"].astype("string").str.strip()
print("Missing beneficiary groups:", merged_df["beneficiary_groups"].isna().sum())
print("Unique beneficiary values:", merged_df["beneficiary_groups"].nunique())

Missing beneficiary groups: 233
Unique beneficiary values: 197


In [41]:
print(merged_df["beneficiary_groups"].value_counts().head(20))

beneficiary_groups
Individual                                                                    3429
Business Entity                                                                231
Family                                                                         119
Business Entity, Industries                                                    118
Individual, Family                                                              90
Family, Individual                                                              51
Registered Societies                                                            36
University / Institution                                                        34
All                                                                             30
Industries                                                                      28
Artists                                                                         26
Individual, Self Help Groups (SHGS)                                 

In [42]:
merged_df["application_mode_unified"] = merged_df["application_mode"].astype("string").str.strip()
print(merged_df["application_mode_unified"].value_counts(dropna=False))

application_mode_unified
<NA>                      2796
Online                     914
Offline                    868
Online & Offline           277
Direct / Not Specified       7
Name: count, dtype: Int64


In [43]:
merged_df["raw_tags_unified"] = merged_df["raw_tags"].astype("string").str.strip()
print("Missing tags:", merged_df["raw_tags_unified"].isna().sum())
print("Unique tag strings:", merged_df["raw_tags_unified"].nunique())

Missing tags: 2812
Unique tag strings: 2021


In [44]:
merged_df["description"] = merged_df["description_hf"].combine_first(merged_df["description_github"])
print("Missing description:", merged_df["description"].isna().sum())

Missing description: 3


In [45]:
merged_df["benefits_unified"] = merged_df["benefits_hf"].combine_first(merged_df["benefits"])
print("Missing benefits:", merged_df["benefits_unified"].isna().sum())

Missing benefits: 3


In [46]:
merged_df["eligibility_text"] = merged_df["eligibility_text_hf"].combine_first(merged_df["eligibility_criteria"]).combine_first(merged_df["eligibility_general"])
print("Missing eligibility text:", merged_df["eligibility_text"].isna().sum())

Missing eligibility text: 4


In [47]:
merged_df["exclusion_text_unified"] = merged_df["exclusion_text"].astype("string").str.strip()
print("Missing exclusion text:", merged_df["exclusion_text_unified"].isna().sum())

Missing exclusion text: 4483


In [48]:
merged_df["application_process_unified"] = merged_df["application_process_hf"].combine_first(merged_df["application_process"])
print("Missing application process:", merged_df["application_process_unified"].isna().sum())

Missing application process: 3


In [49]:
merged_df["documents_required_unified"] = merged_df["documents_required_hf"].combine_first(merged_df["documents_required"])
print("Missing documents:", merged_df["documents_required_unified"].isna().sum())

Missing documents: 17


In [50]:
merged_df["faqs_unified"] = merged_df["faqs"].astype("string").str.strip()
print("Missing FAQs:", merged_df["faqs_unified"].isna().sum())

Missing FAQs: 2796


In [51]:
merged_df["official_url"] = merged_df["official_url_hf"].combine_first(merged_df["official_url_github"])
merged_df["myscheme_url_unified"] = merged_df["myscheme_url"].astype("string").str.strip()
merged_df["apply_url_unified"] = merged_df["apply_url"].astype("string").str.strip()
print("Missing official URL:", merged_df["official_url"].isna().sum())
print("Missing MyScheme URL:", merged_df["myscheme_url_unified"].isna().sum())
print("Missing apply URL:", merged_df["apply_url_unified"].isna().sum())

Missing official URL: 0
Missing MyScheme URL: 2796
Missing apply URL: 178


In [52]:
merged_df["age_min_unified"] = merged_df["age_min"]
merged_df["age_max_unified"] = merged_df["age_max"]
merged_df["income_max_unified"] = merged_df["income_max"]
merged_df["gender_eligibility_unified"] = merged_df["gender_eligibility"].astype("string").str.strip()
merged_df["caste_category_unified"] = merged_df["caste_category"].astype("string").str.strip()
merged_df["residence_requirement_unified"] = merged_df["residence_requirement"].astype("string").str.strip()
merged_df["disability_eligibility_unified"] = merged_df["disability_eligibility"].astype("string").str.strip()
merged_df["bpl_required_unified"] = merged_df["bpl_required"].astype("string").str.strip()
print(merged_df[["age_min_unified","age_max_unified","income_max_unified","gender_eligibility_unified","caste_category_unified","residence_requirement_unified","disability_eligibility_unified","bpl_required_unified"]].isna().sum())

age_min_unified                   3809
age_max_unified                   4077
income_max_unified                4083
gender_eligibility_unified         169
caste_category_unified             169
residence_requirement_unified      169
disability_eligibility_unified     169
bpl_required_unified               169
dtype: int64


In [53]:
print(merged_df[["eligibility_text","age_min_unified","age_max_unified","income_max_unified"]].isna().sum())

eligibility_text         4
age_min_unified       3809
age_max_unified       4077
income_max_unified    4083
dtype: int64


In [54]:
merged_df["scraped_at_unified"] = merged_df["scraped_at"].astype("string").str.strip()
print("Missing scraped_at:", merged_df["scraped_at_unified"].isna().sum())

Missing scraped_at: 169


In [55]:
merged_df["source_hf"] = merged_df["_merge"].isin(["both", "left_only"])
merged_df["source_github"] = merged_df["_merge"].isin(["both", "right_only"])
merged_df["source_count"] = merged_df[["source_hf", "source_github"]].sum(axis=1)
print(merged_df[["source_hf", "source_github", "source_count"]].value_counts())

source_hf  source_github  source_count
True       False          1               2796
           True           2               1897
False      True           1                169
Name: count, dtype: int64


In [56]:
print(merged_df[["source_hf", "source_github", "source_count"]].sum())

source_hf        4693
source_github    2066
source_count     6759
dtype: int64


In [57]:
print(merged_df[["source_hf", "source_github", "source_count"]].value_counts())

source_hf  source_github  source_count
True       False          1               2796
           True           2               1897
False      True           1                169
Name: count, dtype: int64


In [58]:
print(merged_df[["source_hf", "source_github", "source_count"]].value_counts())

source_hf  source_github  source_count
True       False          1               2796
           True           2               1897
False      True           1                169
Name: count, dtype: int64


In [60]:
merged_df["source"] = np.select([merged_df["source_count"].eq(2), merged_df["source_hf"], merged_df["source_github"]], ["HF + GitHub", "HF", "GitHub"], default="Unknown")
print(merged_df["source"].value_counts())

source
HF             2796
HF + GitHub    1897
GitHub          169
Name: count, dtype: int64


In [61]:
print(merged_df[["scheme_id","scheme_name","source"]].head(10))

  scheme_id  \
0  108easuk   
1   15dsugt   
2      1pmy   
3   25-ciss   
4  40shydcs   
5   6islbsa   
6   6islbse   
7  6isltvpb   
8  a-gainer   
9    a-pudu   

                                                                                       scheme_name  \
0                                                   108, Emergency Ambulance Service - Uttarakhand   
1                                                              15 Days Skill Up-gradation Training   
2                                                                         100% Penalty Mafi Yojana   
3                                                            25% Capital Investment Subsidy Scheme   
4                                                40% Subsidy On Hank Yarn, Dyes & Chemicals Scheme   
5                                6% Interest Subsidy on Loans taken through Banks for Study Abroad   
6  6% Interest Subsidy on Loans taken through Banks up to a Limit of Rs.5 Lakh for Self-Employment   
7                 

In [62]:
merged_df["combined_text"] = merged_df[["scheme_name","description","benefits_unified","eligibility_text","exclusion_text_unified","application_process_unified","documents_required_unified","faqs_unified"]].fillna("").astype(str).apply(lambda x: "\n".join(f"{col}: {val}" for col, val in zip(["Scheme Name","Description","Benefits","Eligibility","Exclusions","Application Process","Documents Required","FAQs"], x) if val.strip()), axis=1)
print("Missing combined_text:", merged_df["combined_text"].eq("").sum())
print("Average combined_text length:", merged_df["combined_text"].str.len().mean())

Missing combined_text: 0
Average combined_text length: 4459.427807486631


In [63]:
print(merged_df[["scheme_id","scheme_name","combined_text"]].sample(5, random_state=42).to_string(index=False))

scheme_id                                                                                                                                                                                                                                          scheme_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [64]:
print(merged_df[["scheme_id","scheme_name","combined_text"]].sample(5, random_state=42).to_string(index=False))

scheme_id                                                                                                                                                                                                                                          scheme_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [65]:
import html
merged_df["combined_text"] = merged_df["combined_text"].apply(lambda x: html.unescape(str(x)).replace("\\n", "\n").replace("/n", "\n").replace("<br>", "\n").strip())
print("Average combined_text length:", merged_df["combined_text"].str.len().mean())
print("Empty combined_text:", merged_df["combined_text"].eq("").sum())

Average combined_text length: 4452.11065405183
Empty combined_text: 0


In [66]:
print(merged_df[["scheme_id","scheme_name","combined_text"]].sample(5, random_state=42).to_string(index=False))

scheme_id                                                                                                                                                                                                                                          scheme_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                          

In [67]:
import re
merged_df["combined_text"] = merged_df["combined_text"].apply(lambda x: re.sub(r"\\([*_`])", r"\1", str(x)).replace("\\", "").replace("&#39;", "'").replace("&#x20;", " ").strip())

In [68]:
print(merged_df["combined_text"].str.len().describe())
print("Empty combined_text:", merged_df["combined_text"].eq("").sum())
print("Missing combined_text:", merged_df["combined_text"].isna().sum())

count     4862.000000
mean      4451.698478
std       2862.959763
min         42.000000
25%       2405.000000
50%       3776.500000
75%       5795.750000
max      32013.000000
Name: combined_text, dtype: float64
Empty combined_text: 0
Missing combined_text: 0


In [69]:
print(merged_df[["scheme_id","scheme_name","combined_text"]].sort_values("combined_text", key=lambda x: x.str.len()).head(10).to_string(index=False))

scheme_id                                                                       scheme_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      combined_text
     vpby                                                     Varishtha Pension Bima Yojana                                                                              

In [70]:
print(merged_df[["scheme_id","scheme_name","description","benefits_unified","eligibility_text","application_process_unified","documents_required_unified"]].isna().sum())

scheme_id                       0
scheme_name                     0
description                     3
benefits_unified                3
eligibility_text                4
application_process_unified     3
documents_required_unified     17
dtype: int64


In [71]:
print(merged_df.loc[merged_df[["description","benefits_unified","eligibility_text","application_process_unified","documents_required_unified"]].isna().any(axis=1), ["scheme_id","scheme_name","description","benefits_unified","eligibility_text","application_process_unified","documents_required_unified"]].to_string(index=False))

                                  scheme_id                                                                                                                                                                                             scheme_name                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

In [72]:
print(merged_df[["description","benefits_unified","eligibility_text","application_process_unified","documents_required_unified"]].isna().sum())

description                     3
benefits_unified                3
eligibility_text                4
application_process_unified     3
documents_required_unified     17
dtype: int64


In [73]:
print(merged_df.loc[merged_df[["description","benefits_unified","eligibility_text","application_process_unified","documents_required_unified"]].isna().any(axis=1), ["scheme_id","scheme_name","source"]].to_string(index=False))

                                  scheme_id                                                                                                                                                                                             scheme_name      source
                                        cia                                                                                                                                              Cash Incentive Award to Bonafede Meritorious Sportspersons          HF
                                       eesd                                                                                                                                                                  Endeavour To Empower-Skill Development          HF
                                  fafmftcwd                                                                                                                                           Financial assistance for Marriage for the Construc

In [74]:
missing_cols = ["description","benefits_unified","eligibility_text","application_process_unified","documents_required_unified"]
print(merged_df.loc[merged_df[missing_cols].isna().any(axis=1), ["scheme_id","scheme_name","source"] + missing_cols].to_string(index=False))

                                  scheme_id                                                                                                                                                                                             scheme_name      source                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [75]:
missing_cols = ["description","benefits_unified","eligibility_text","application_process_unified","documents_required_unified"]
print(merged_df.loc[merged_df[missing_cols].isna().any(axis=1), ["scheme_id","scheme_name","source"] + missing_cols].notna().sum())

scheme_id                      18
scheme_name                    18
source                         18
description                    15
benefits_unified               15
eligibility_text               14
application_process_unified    15
documents_required_unified      1
dtype: int64


In [76]:
print(merged_df.loc[merged_df[missing_cols].isna().any(axis=1), ["scheme_id","scheme_name"]].to_string(index=False))

                                  scheme_id                                                                                                                                                                                             scheme_name
                                        cia                                                                                                                                              Cash Incentive Award to Bonafede Meritorious Sportspersons
                                       eesd                                                                                                                                                                  Endeavour To Empower-Skill Development
                                  fafmftcwd                                                                                                                                           Financial assistance for Marriage for the Construction Worker
                        

In [77]:
print(merged_df.loc[merged_df["description"].isna() | merged_df["benefits_unified"].isna() | merged_df["eligibility_text"].isna() | merged_df["application_process_unified"].isna(), ["scheme_id","scheme_name","source"]].to_string(index=False))

scheme_id                                                                       scheme_name source
fafmftcwd                     Financial assistance for Marriage for the Construction Worker     HF
     mrin                                               Market Research Information Network GitHub
  visvasi Vanchit Ikai Samooh aur Vargon ki Aarthik Sahayata Yojana (VISVAS) for Individual     HF
     vpby                                                     Varishtha Pension Bima Yojana     HF


In [78]:
cols = ["scheme_id","scheme_name","description_hf","description_github","benefits_hf","benefits","eligibility_text_hf","eligibility_criteria","eligibility_general","application_process_hf","application_process"]
print(merged_df.loc[merged_df["scheme_id"].isin(["fafmftcwd","mrin","visvasi","vpby"]), cols].T.to_string())

                                                                                 1229                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                   

In [79]:
text_cols = ["description","benefits_unified","eligibility_text","exclusion_text_unified","application_process_unified","documents_required_unified","faqs_unified"]
merged_df[text_cols] = merged_df[text_cols].fillna("")

In [80]:
text_map = {"Scheme Name":"scheme_name","Government Level":"government_level_unified","State/UT":"state_ut","Ministry":"ministry","Department":"department_unified","Category":"category_unified","Beneficiary":"beneficiary_groups","Application Mode":"application_mode_unified","Description":"description","Benefits":"benefits_unified","Eligibility":"eligibility_text","Exclusions":"exclusion_text_unified","Application Process":"application_process_unified","Documents Required":"documents_required_unified","FAQs":"faqs_unified"}
merged_df["combined_text"] = merged_df[list(text_map.values())].fillna("").astype(str).apply(lambda x: "\n".join(f"{k}: {v}" for k,v in zip(text_map.keys(),x) if v.strip() and v.strip().lower() != "nan"), axis=1)
print(merged_df["combined_text"].str.len().describe())

count     4862.000000
mean      4636.082065
std       2870.653174
min         68.000000
25%       2573.000000
50%       3959.500000
75%       5984.750000
max      32277.000000
Name: combined_text, dtype: float64


In [82]:
print(merged_df.loc[merged_df["combined_text"].str.len().sort_values().head(15).index, ["scheme_id","scheme_name","source","combined_text"]].to_string(index=False))

scheme_id                                                                       scheme_name source                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [83]:
print("Duplicate combined_text:", merged_df["combined_text"].duplicated().sum())

Duplicate combined_text: 0


In [84]:
print("Rows:", len(merged_df))
print("Unique scheme_id:", merged_df["scheme_id"].nunique())
print("Duplicate scheme_id:", merged_df["scheme_id"].duplicated().sum())
print("Missing scheme_id:", merged_df["scheme_id"].isna().sum())
print("Unique scheme_name:", merged_df["scheme_name"].nunique())
print("Duplicate scheme_name:", merged_df["scheme_name"].duplicated().sum())
print("Missing scheme_name:", merged_df["scheme_name"].isna().sum())

Rows: 4862
Unique scheme_id: 4862
Duplicate scheme_id: 0
Missing scheme_id: 0
Unique scheme_name: 4862
Duplicate scheme_name: 0
Missing scheme_name: 0


In [85]:
print(merged_df[["source_hf","source_github","source_count","source"]].value_counts(dropna=False).sort_index())
print("\nSource count totals:", merged_df["source_count"].sum())
print("Invalid source_count:", (~merged_df["source_count"].isin([1,2])).sum())

source_hf  source_github  source_count  source     
False      True           1             GitHub          169
True       False          1             HF             2796
           True           2             HF + GitHub    1897
Name: count, dtype: int64

Source count totals: 6759
Invalid source_count: 0


In [86]:
final_candidate_cols = ["scheme_id","scheme_name","scheme_slug","government_level_unified","state_ut","ministry","department_unified","category_unified","raw_tags_unified","beneficiary_groups","age_min_unified","age_max_unified","income_max_unified","gender_eligibility_unified","caste_category_unified","residence_requirement_unified","disability_eligibility_unified","bpl_required_unified","eligibility_text","exclusion_text_unified","description","benefits_unified","application_mode_unified","application_process_unified","documents_required_unified","faqs_unified","official_url","myscheme_url_unified","apply_url_unified","source_hf","source_github","source_count","source","scraped_at_unified","combined_text"]
print(merged_df[final_candidate_cols].isna().sum().sort_values(ascending=False).to_string())

ministry                          4098
income_max_unified                4083
age_max_unified                   4077
age_min_unified                   3809
raw_tags_unified                  2812
myscheme_url_unified              2796
application_mode_unified          2796
state_ut                           764
department_unified                 462
beneficiary_groups                 233
apply_url_unified                  178
category_unified                   173
disability_eligibility_unified     169
gender_eligibility_unified         169
scraped_at_unified                 169
bpl_required_unified               169
residence_requirement_unified      169
caste_category_unified             169
scheme_id                            0
scheme_slug                          0
scheme_name                          0
eligibility_text                     0
government_level_unified             0
application_process_unified          0
exclusion_text_unified               0
description              

In [87]:
print("Government level:")
print(merged_df["government_level_unified"].value_counts(dropna=False).to_string())
print("\nState/UT samples:")
print(merged_df["state_ut"].dropna().value_counts().head(30).to_string())

Government level:
government_level_unified
State/UT    4098
Central      764

State/UT samples:
state_ut
Gujarat                                 639
Uttarakhand                             448
Madhya Pradesh                          286
Goa                                     272
Puducherry                              270
Haryana                                 246
Tamil Nadu                              232
Rajasthan                               158
Bihar                                   113
West Bengal                             112
Chhattisgarh                            107
Jharkhand                                95
Odisha                                   89
Maharashtra                              85
Meghalaya                                84
Assam                                    82
Kerala                                   81
Himachal Pradesh                         71
Andhra Pradesh                           65
Karnataka                                64
Dadra & Nagar H

In [88]:
print(merged_df.loc[merged_df["state_ut"].notna() & merged_df["state_ut"].str.contains("Dadra|Daman|Nagar", case=False, regex=True), "state_ut"].value_counts().to_string())

state_ut
Dadra & Nagar Haveli and Daman & Diu        61
Andaman and Nicobar Islands                 19
Dadra and Nagar Haveli and Daman and Diu     1


In [89]:
merged_df["state_ut"] = merged_df["state_ut"].replace({"Dadra and Nagar Haveli and Daman and Diu": "Dadra & Nagar Haveli and Daman & Diu"})
print(merged_df["state_ut"].dropna().value_counts().loc[lambda x: x.index.str.contains("Dadra|Daman|Nagar", case=False, regex=True)].to_string())

state_ut
Dadra & Nagar Haveli and Daman & Diu    62
Andaman and Nicobar Islands             19


In [90]:
print(merged_df["category_unified"].dropna().value_counts().to_string())

category_unified
Education & Learning                                                                                                                                                            727
Social Welfare & Empowerment                                                                                                                                                    686
Agriculture & Rural Development                                                                                                                                                 667
Business & Entrepreneurship                                                                                                                                                     417
Social Welfare & Empowerment, Women and Child                                                                                                                                   155
Sports & Culture                                                                   

In [91]:
print(merged_df["beneficiary_groups"].dropna().value_counts().head(50).to_string())

beneficiary_groups
Individual                                                                                                                                        3429
Business Entity                                                                                                                                    231
Family                                                                                                                                             119
Business Entity, Industries                                                                                                                        118
Individual, Family                                                                                                                                  90
Family, Individual                                                                                                                                  51
Registered Societies                                                       

In [92]:
b = merged_df["beneficiary_groups"].dropna().str.split(", ").apply(lambda x: tuple(sorted(x)))
print(b.value_counts().head(30).to_string())

beneficiary_groups
(Individual,)                                                                                                                                       3429
(Business Entity,)                                                                                                                                   231
(Family, Individual)                                                                                                                                 141
(Business Entity, Industries)                                                                                                                        125
(Family,)                                                                                                                                            119
(Business Entity, Individual)                                                                                                                         39
(Registered Societies,)                                        

In [93]:
merged_df["beneficiary_groups_normalized"] = merged_df["beneficiary_groups"].apply(lambda x: ", ".join(sorted(str(x).split(", "))) if pd.notna(x) else pd.NA)
print(merged_df["beneficiary_groups_normalized"].value_counts().head(30).to_string())

beneficiary_groups_normalized
Individual                                                                                                                                        3429
Business Entity                                                                                                                                    231
Family, Individual                                                                                                                                 141
Business Entity, Industries                                                                                                                        125
Family                                                                                                                                             119
Business Entity, Individual                                                                                                                         39
Registered Societies                                            

In [94]:
merged_df["categories"] = merged_df["category_unified"].apply(lambda x: ", ".join(sorted(str(x).split(", "))) if pd.notna(x) else pd.NA)
print(merged_df["categories"].value_counts().head(30).to_string())

categories
Education & Learning                                                        727
Social Welfare & Empowerment                                                686
Agriculture & Rural Development                                             667
Business & Entrepreneurship                                                 417
Social Welfare & Empowerment, Women and Child                               188
Sports & Culture                                                            154
Education & Learning, Social Welfare & Empowerment                          129
Skills & Employment                                                         127
Health & Wellness                                                           124
Banking, Financial Services & Insurance                                      72
Banking, Business & Entrepreneurship, Financial Services & Insurance         71
Housing & Shelter                                                            65
Health & Wellness, Social Wel

In [95]:
merged_df["application_mode"] = merged_df["application_mode_unified"].replace({"Online & Offline": "Online + Offline", "Direct / Not Specified": "Direct / Not Specified"}).astype("string").str.strip()
print(merged_df["application_mode"].value_counts(dropna=False).to_string())

application_mode
<NA>                      2796
Online                     914
Offline                    868
Online + Offline           277
Direct / Not Specified       7


In [96]:
merged_df["tags_normalized"] = merged_df["raw_tags_unified"].apply(lambda x: ", ".join(sorted({t.strip() for t in str(x).split(",") if t.strip()})) if pd.notna(x) and str(x).strip() else pd.NA)
print(merged_df["tags_normalized"].dropna().head(10).to_string(index=False))

                                       Capital, Entrepreneur, Investment, Subsidy, Unemployed, Youth
                                              Chemicals, Dyes, Handloom, Hank, Subsidy, Weaver, Yarn
                               AICTE, Augmenting, Eastern, For, Grant, Infrastructure, North, Region
                                               Assistance, Child, Financial, Girl, Mother, Nutrition
Access, And, Development, Dissemination, For, Industry, Institution, Knowledge, Ministry, Proposa...
                               Below, Cashless, Health, Hospital, Line, Patients, Poverty, Treatment
AABCS, Ambedkar, Annal, Business, Caste, Champions, Entrepreneur, Loan, MSME, Scheduled, Subsidy,...
                                                              Bima, DBT, Incentive, Insurance, Labor
                                                       Abhinandan, Education, Loan, Student, Subsidy
                                                                                       Loan

In [97]:
print("Missing:", merged_df["tags_normalized"].isna().sum())
print("Unique:", merged_df["tags_normalized"].nunique())
print(merged_df["tags_normalized"].dropna().head(20).to_string(index=False))

Missing: 2812
Unique: 2020
                                       Capital, Entrepreneur, Investment, Subsidy, Unemployed, Youth
                                              Chemicals, Dyes, Handloom, Hank, Subsidy, Weaver, Yarn
                               AICTE, Augmenting, Eastern, For, Grant, Infrastructure, North, Region
                                               Assistance, Child, Financial, Girl, Mother, Nutrition
Access, And, Development, Dissemination, For, Industry, Institution, Knowledge, Ministry, Proposa...
                               Below, Cashless, Health, Hospital, Line, Patients, Poverty, Treatment
AABCS, Ambedkar, Annal, Business, Caste, Champions, Entrepreneur, Loan, MSME, Scheduled, Subsidy,...
                                                              Bima, DBT, Incentive, Insurance, Labor
                                                       Abhinandan, Education, Loan, Student, Subsidy
                                                                

In [98]:
eligibility_parts = ["eligibility_text","age_min_unified","age_max_unified","income_max_unified","gender_eligibility_unified","caste_category_unified","residence_requirement_unified","disability_eligibility_unified","bpl_required_unified"]
eligibility_labels = ["Eligibility","Minimum Age","Maximum Age","Maximum Income","Gender","Caste Category","Residence Requirement","Disability","BPL"]
merged_df["eligibility_text_final"] = merged_df[eligibility_parts].fillna("").astype(str).apply(lambda x: "\n".join(f"{k}: {v}" for k,v in zip(eligibility_labels,x) if v.strip() and v.strip().lower() != "nan"), axis=1)
print(merged_df["eligibility_text_final"].str.len().describe())

count    4862.000000
mean      778.279309
std       687.174286
min         0.000000
25%       391.250000
50%       589.500000
75%       912.000000
max      9348.000000
Name: eligibility_text_final, dtype: float64


In [99]:
text_map = {"Scheme Name":"scheme_name","Government Level":"government_level_unified","State/UT":"state_ut","Ministry":"ministry","Department":"department_unified","Category":"categories","Beneficiary":"beneficiary_groups_normalized","Application Mode":"application_mode","Description":"description","Benefits":"benefits_unified","Eligibility":"eligibility_text_final","Exclusions":"exclusion_text_unified","Application Process":"application_process_unified","Documents Required":"documents_required_unified","FAQs":"faqs_unified","Tags":"tags_normalized"}
merged_df["combined_text"] = merged_df[list(text_map.values())].fillna("").astype(str).apply(lambda x: "\n".join(f"{k}: {v}" for k,v in zip(text_map.keys(),x) if v.strip() and v.strip().lower() != "nan"), axis=1)
print(merged_df["combined_text"].str.len().describe())

count     4862.000000
mean      4785.122995
std       2895.921508
min        169.000000
25%       2708.250000
50%       4112.500000
75%       6150.500000
max      32473.000000
Name: combined_text, dtype: float64


In [100]:
print(merged_df.columns.tolist())

['scheme_slug_hf', 'scheme_name_hf', 'description_hf', 'ministry_hf', 'department', 'state_hf', 'category_hf', 'beneficiary_type_hf', 'benefits_hf', 'eligibility_text_hf', 'application_process_hf', 'documents_required_hf', 'apply_url', 'official_url_hf', 'age_min', 'age_max', 'gender_eligibility', 'caste_category', 'income_max', 'residence_requirement', 'eligibility_state', 'disability_eligibility', 'bpl_required', 'scraped_at', 'name_normalized_hf', 'scheme_id_hf', 'unified_scheme_id', 'scheme_name_github', 'scheme_slug_github', 'government_level', 'state_ut_ministry', 'application_mode', 'raw_tags', 'description_github', 'eligibility_criteria', 'eligibility_general', 'exclusion_text', 'benefits', 'application_process', 'documents_required', 'faqs', 'official_url_github', 'myscheme_url', 'name_normalized_github', 'scheme_id_github', '_merge', 'scheme_name', 'scheme_slug', 'scheme_id', 'government_level_unified', 'state_ut', 'ministry', 'department_unified', 'category_unified', 'benefi

In [101]:
final_cols = ["scheme_id","scheme_name","scheme_slug","government_level_unified","state_ut","ministry","department_unified","categories","beneficiary_groups_normalized","age_min_unified","age_max_unified","income_max_unified","gender_eligibility_unified","caste_category_unified","residence_requirement_unified","disability_eligibility_unified","bpl_required_unified","eligibility_text_final","exclusion_text_unified","description","benefits_unified","application_mode","application_process_unified","documents_required_unified","faqs_unified","tags_normalized","official_url","myscheme_url_unified","apply_url_unified","source_hf","source_github","source_count","source","scraped_at_unified","combined_text"]
print("Final columns:", len(final_cols))
print("\nMissing columns:", [c for c in final_cols if c not in merged_df.columns])

Final columns: 35

Missing columns: []


In [102]:
final_df = merged_df[final_cols].copy()
print(final_df.shape)
print(final_df.head(2).to_string())

(4862, 35)
  scheme_id                                     scheme_name scheme_slug government_level_unified        state_ut ministry                            department_unified           categories beneficiary_groups_normalized  age_min_unified  age_max_unified  income_max_unified gender_eligibility_unified caste_category_unified residence_requirement_unified disability_eligibility_unified bpl_required_unified                                                                                                                                                                                                                                                    eligibility_text_final exclusion_text_unified                                                                                                                                                                                                                                           description                                                  

In [103]:
core_cols = ["scheme_id","scheme_name","government_level_unified","description","benefits_unified","eligibility_text_final","official_url","combined_text"]
print("Shape:", final_df.shape)
print("Duplicate IDs:", final_df["scheme_id"].duplicated().sum())
print("Duplicate names:", final_df["scheme_name"].duplicated().sum())
print("Duplicate combined_text:", final_df["combined_text"].duplicated().sum())
print("Missing core fields:\n", final_df[core_cols].isna().sum().to_string())

Shape: (4862, 35)
Duplicate IDs: 0
Duplicate names: 0
Duplicate combined_text: 0
Missing core fields:
 scheme_id                   0
scheme_name                 0
government_level_unified    0
description                 0
benefits_unified            0
eligibility_text_final      0
official_url                0
combined_text               0


In [104]:
print(final_df.dtypes.to_string())

scheme_id                                 object
scheme_name                               object
scheme_slug                               object
government_level_unified                  object
state_ut                          string[python]
ministry                          string[python]
department_unified                string[python]
categories                                object
beneficiary_groups_normalized             object
age_min_unified                          float64
age_max_unified                          float64
income_max_unified                       float64
gender_eligibility_unified        string[python]
caste_category_unified            string[python]
residence_requirement_unified     string[python]
disability_eligibility_unified    string[python]
bpl_required_unified              string[python]
eligibility_text_final                    object
exclusion_text_unified            string[python]
description                               object
benefits_unified    

In [105]:
final_df.to_csv("/content/government_schemes_final.csv", index=False)
print("Saved:", final_df.shape)

Saved: (4862, 35)
